In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece ftfy
!pip uninstall -y -q torchao

In [ ]:
import torch, time, gc, json
from diffusers.utils import export_to_video
PROMPT = ("A giant majestic dragon hatching from a glowing egg in a dark volcanic cave, "
          "cinematic lighting, sparks and smoke, the baby dragon slowly opens its eyes and spreads small wings, "
          "highly detailed, realistic, movie scene")
NEG = "blurry, low quality, distorted, watermark, text"
RESULTS = {}
def free():
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Test 1: LTX-Video (2B) - 4 second clip
try:
    from diffusers import LTXPipeline
    t0 = time.time()
    pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload(); pipe.vae.enable_tiling()
    load = time.time() - t0
    for n in range(2):
        t0 = time.time()
        frames = pipe(prompt=PROMPT, negative_prompt=NEG, width=704, height=480, num_frames=97,
                      num_inference_steps=30, generator=torch.Generator().manual_seed(n)).frames[0]
        dt = time.time() - t0
        export_to_video(frames, f"ltx_{n}.mp4", fps=24)
        frames[48].save(f"ltx_{n}.png")
        RESULTS[f"ltx_clip{n}_sec"] = round(dt)
        print(f"LTX clip {n}: {dt:.0f} sec", flush=True)
    RESULTS["ltx_load_sec"] = round(load)
    del pipe; free()
except Exception as e:
    RESULTS["ltx_error"] = str(e)[:500]; print("LTX ERROR", e)

In [ ]:
# Test 2: Wan 2.1 (1.3B) - ~3 second clip 480p
try:
    from diffusers import WanPipeline, AutoencoderKLWan
    mid = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
    t0 = time.time()
    vae = AutoencoderKLWan.from_pretrained(mid, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(mid, vae=vae, torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    load = time.time() - t0
    t0 = time.time()
    frames = pipe(prompt=PROMPT, negative_prompt=NEG, width=832, height=480, num_frames=49,
                  num_inference_steps=30, guidance_scale=5.0,
                  generator=torch.Generator().manual_seed(0)).frames[0]
    dt = time.time() - t0
    export_to_video(frames, "wan_0.mp4", fps=16)
    import PIL.Image, numpy as np
    PIL.Image.fromarray((np.array(frames[24]) * 255).astype("uint8") if np.array(frames[24]).max() <= 1 else np.array(frames[24])).save("wan_0.png")
    RESULTS["wan_load_sec"] = round(load); RESULTS["wan_clip0_sec"] = round(dt)
    print(f"Wan clip: {dt:.0f} sec", flush=True)
    del pipe; free()
except Exception as e:
    RESULTS["wan_error"] = str(e)[:500]; print("WAN ERROR", e)

In [ ]:
json.dump(RESULTS, open("results.json", "w"), indent=1)
print(json.dumps(RESULTS, indent=1))
from IPython.display import Video, display
import os
for f in ["ltx_0.mp4", "ltx_1.mp4", "wan_0.mp4"]:
    if os.path.exists(f):
        print(f); display(Video(f, embed=True, width=480))